# 05a — Pipeline Funnel Diagnostics

Tracks exactly where users drop at each stage between NB03 labeling and the NB04 analysis panel.
Answers the question: *of 2,871 exposed users, why do only 967 make it into the panel?*

**Funnel stages:**
1. Labeled by NB03 (exposure_labels_v2)
2. Has any pre-period activity (Sep–Nov before first anchor comment)
3. Has any post-period activity (Dec–May)
4. Has **both** pre + post → enters analysis panel

**Dropout taxonomy (exposed users):**
- `no_activity` — no pre or post observations (lurkers / deleted accounts)
- `post_only` — post-period activity but no pre-period baseline
- `pre_only` — pre-period activity but no post-period outcome
- `in_panel` — both pre + post present

**Inputs:** `exposure_labels.parquet` (NB03), `panel_scores.parquet` (NB04), `post_level_scores.parquet` (NB04)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SUBREDDIT = 'gradadmissions'  # change to

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed' / SUBREDDIT
FIG_DIR  = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

labels   = pd.read_parquet(DATA_DIR / 'exposure_labels.parquet')
panel    = pd.read_parquet(DATA_DIR / 'panel_scores.parquet')
post_lvl = pd.read_parquet(DATA_DIR / 'post_level_scores.parquet')

print(f'Labels:   {len(labels):,} rows | exposed={labels["exposed"].sum():,} | unexposed={(~labels["exposed"]).sum():,}')
print(f'Panel:    {len(panel):,} rows | unique users={panel["author"].nunique():,}')
print(f'Post-lvl: {len(post_lvl):,} rows')

## 1) Build per-(author, cycle) activity flags

In [ ]:
# Which (author, cycle) pairs have pre and/or post observations in post_level_scores
has_pre = (
    post_lvl[post_lvl['window'] == 'pre']
    .groupby(['author', 'cycle'])
    .size()
    .reset_index(name='pre_n')
    .assign(has_pre=True)
)

has_post = (
    post_lvl[post_lvl['window'] == 'post']
    .groupby(['author', 'cycle'])
    .size()
    .reset_index(name='post_n')
    .assign(has_post=True)
)

# Merge flags onto all labeled users
funnel = (
    labels[['author', 'cycle', 'exposed']]
    .merge(has_pre[['author', 'cycle', 'has_pre', 'pre_n']],   on=['author', 'cycle'], how='left')
    .merge(has_post[['author', 'cycle', 'has_post', 'post_n']], on=['author', 'cycle'], how='left')
)
funnel['has_pre']  = funnel['has_pre'].fillna(False)
funnel['has_post'] = funnel['has_post'].fillna(False)
funnel['pre_n']    = funnel['pre_n'].fillna(0).astype(int)
funnel['post_n']   = funnel['post_n'].fillna(0).astype(int)

# Four-way classification
def classify(row):
    if row['has_pre'] and row['has_post']:
        return 'in_panel'
    elif row['has_post'] and not row['has_pre']:
        return 'post_only'
    elif row['has_pre'] and not row['has_post']:
        return 'pre_only'
    else:
        return 'no_activity'

funnel['fate'] = funnel.apply(classify, axis=1)

print('Overall fate distribution:')
print(funnel.groupby(['exposed', 'fate']).size().unstack(fill_value=0))

## 2) Exposed-user funnel table

In [ ]:
exposed = funnel[funnel['exposed']].copy()
n_total = len(exposed)

fate_order  = ['in_panel', 'post_only', 'pre_only', 'no_activity']
fate_labels = {
    'in_panel':    'Has both pre + post  →  enters panel',
    'post_only':   'Has post only  (no pre-period baseline)',
    'pre_only':    'Has pre only   (no post-period outcome)',
    'no_activity': 'Has neither pre nor post  (silent / lurker)',
}

rows = []
for fate in fate_order:
    n = (exposed['fate'] == fate).sum()
    rows.append({'Fate': fate_labels[fate], 'N': n, '%': f'{n/n_total*100:.1f}%'})

tbl = pd.DataFrame(rows)
tbl.loc[len(tbl)] = ['TOTAL EXPOSED', n_total, '100.0%']
print(f'Exposed users: {n_total:,}')
print()
print(tbl.to_string(index=False))

## 3) Unexposed-user funnel table (for comparison)

In [ ]:
unexposed  = funnel[~funnel['exposed']].copy()
n_unexp    = len(unexposed)

rows_u = []
for fate in fate_order:
    n = (unexposed['fate'] == fate).sum()
    rows_u.append({'Fate': fate_labels[fate], 'N': n, '%': f'{n/n_unexp*100:.1f}%'})

tbl_u = pd.DataFrame(rows_u)
tbl_u.loc[len(tbl_u)] = ['TOTAL UNEXPOSED', n_unexp, '100.0%']
print(f'Unexposed users: {n_unexp:,}')
print()
print(tbl_u.to_string(index=False))

## 4) Waterfall chart — exposed user dropout

In [ ]:
fate_counts = exposed['fate'].value_counts().reindex(fate_order).fillna(0).astype(int)

# Stage counts along the funnel
stages = [
    ('Labeled\n(NB03)',          n_total),
    ('Has any\nactivity',        int(exposed['has_pre'].sum() + exposed['has_post'].sum() - fate_counts['in_panel'])),
    ('Has\npre-period',          int(exposed['has_pre'].sum())),
    ('Has\npost-period',         int(exposed['has_post'].sum())),
    ('Both pre+post\n(panel)',   int(fate_counts['in_panel'])),
]
# 'Has any activity' is everyone except no_activity
stages[1] = ('Has any\nactivity', n_total - int(fate_counts['no_activity']))

labels_s = [s[0] for s in stages]
counts_s = [s[1] for s in stages]
pcts_s   = [c / n_total * 100 for c in counts_s]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4C72B0', '#4C72B0', '#4C72B0', '#4C72B0', '#2ca02c']
bars = ax.bar(labels_s, counts_s, color=colors, alpha=0.8, edgecolor='white', linewidth=1.2)

for bar, n, pct in zip(bars, counts_s, pcts_s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{n:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Dropout annotations between bars
dropout_labels = [
    f'−{int(fate_counts["no_activity"]):,}\nno activity',
    None,
    f'−{int(fate_counts["post_only"]):,}\nno pre',
    f'−{int(fate_counts["pre_only"]):,}\nno post',
]
for i, dlabel in enumerate(dropout_labels):
    if dlabel:
        mid_x = i + 0.5
        mid_y = (counts_s[i] + counts_s[i+1]) / 2
        ax.annotate(dlabel, xy=(mid_x, mid_y), ha='center', va='center',
                    fontsize=8, color='firebrick',
                    bbox=dict(boxstyle='round,pad=0.2', fc='#fff0f0', ec='firebrick', alpha=0.8))

ax.set_ylabel('Number of exposed users')
ax.set_title(f'Pipeline funnel — exposed users (N={n_total:,})', fontsize=12, fontweight='bold')
ax.set_ylim(0, n_total * 1.18)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_pipeline_funnel.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/fig_pipeline_funnel.png')

## 5) Side-by-side: exposed vs unexposed dropout rates

In [ ]:
def fate_pcts(df):
    n = len(df)
    return {fate: (df['fate'] == fate).sum() / n * 100 for fate in fate_order}

exp_pcts  = fate_pcts(exposed)
unexp_pcts = fate_pcts(unexposed)

x = np.arange(len(fate_order))
w = 0.35
short_labels = ['In panel\n(pre+post)', 'Post only\n(no pre)', 'Pre only\n(no post)', 'No activity']
palette = {'exposed': '#d62728', 'unexposed': '#1f77b4'}

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [exp_pcts[f]   for f in fate_order], w, label='Exposed',   color=palette['exposed'],   alpha=0.8)
b2 = ax.bar(x + w/2, [unexp_pcts[f] for f in fate_order], w, label='Unexposed', color=palette['unexposed'], alpha=0.8)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}%',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(short_labels)
ax.set_ylabel('% of users in group')
ax.set_title('Panel fate by exposure status — differential attrition check', fontsize=11, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_funnel_exposed_vs_unexposed.png', dpi=150, bbox_inches='tight')
plt.show()

print('Differential attrition (panel retention):')
print(f'  Exposed   in-panel rate: {exp_pcts["in_panel"]:.1f}%')
print(f'  Unexposed in-panel rate: {unexp_pcts["in_panel"]:.1f}%')
diff = exp_pcts['in_panel'] - unexp_pcts['in_panel']
print(f'  Difference: {diff:+.1f} pp  => {"CLEAN (|diff| < 3pp)" if abs(diff) < 3 else "FLAG — differential attrition"}')

## 6) Per-cycle breakdown

In [ ]:
print('Exposed users — fate by cycle:')
by_cycle_exp = (
    exposed.groupby(['cycle', 'fate'])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=fate_order, fill_value=0)
)
cycle_totals = by_cycle_exp.sum(axis=1)
by_cycle_pct = (by_cycle_exp.div(cycle_totals, axis=0) * 100).round(1)

display_tbl = by_cycle_exp.copy().astype(str)
for col in fate_order:
    display_tbl[col] = by_cycle_exp[col].astype(str) + ' (' + by_cycle_pct[col].astype(str) + '%)'
display_tbl.columns = [fate_labels[c].split('  ')[0].strip() for c in fate_order]
display_tbl.insert(0, 'Total', cycle_totals)
display(display_tbl)

print()
print('Unexposed users — fate by cycle:')
by_cycle_unexp = (
    unexposed.groupby(['cycle', 'fate'])
             .size()
             .unstack(fill_value=0)
             .reindex(columns=fate_order, fill_value=0)
)
unexp_totals = by_cycle_unexp.sum(axis=1)
by_cycle_unexp_pct = (by_cycle_unexp.div(unexp_totals, axis=0) * 100).round(1)
display_tbl_u = by_cycle_unexp.copy().astype(str)
for col in fate_order:
    display_tbl_u[col] = by_cycle_unexp[col].astype(str) + ' (' + by_cycle_unexp_pct[col].astype(str) + '%)'
display_tbl_u.columns = [fate_labels[c].split('  ')[0].strip() for c in fate_order]
display_tbl_u.insert(0, 'Total', unexp_totals)
display(display_tbl_u)

## 7) Activity volume for dropped users

Are the dropped exposed users truly silent, or do they have thin activity that just misses the window boundaries?

In [ ]:
print('Post-period activity for exposed users by fate group:')
print(exposed.groupby('fate')['post_n'].describe()[['count','mean','50%','max']].round(1))

print()
print('Pre-period activity for exposed users by fate group:')
print(exposed.groupby('fate')['pre_n'].describe()[['count','mean','50%','max']].round(1))

## 8) Root-cause summary of the 1,904 excluded exposed users

In [ ]:
n_panel      = int(fate_counts['in_panel'])
n_no_act     = int(fate_counts['no_activity'])
n_post_only  = int(fate_counts['post_only'])
n_pre_only   = int(fate_counts['pre_only'])
n_dropped    = n_total - n_panel

in_panel_rate_exp   = n_panel / n_total * 100
in_panel_rate_unexp = (unexposed['fate'] == 'in_panel').sum() / len(unexposed) * 100

print('=' * 65)
print('PIPELINE FUNNEL SUMMARY — EXPOSED USERS')
print('=' * 65)
print(f'  Labeled by NB03:              {n_total:>6,}')
print(f'  → Enters panel (pre+post):    {n_panel:>6,}  ({in_panel_rate_exp:.1f}%)')
print(f'  → Dropped (total):            {n_dropped:>6,}  ({100-in_panel_rate_exp:.1f}%)')
print()
print('  Dropout breakdown:')
print(f'    No activity at all:         {n_no_act:>6,}  ({n_no_act/n_dropped*100:.1f}% of dropouts)')
print(f'    Post only (missing pre):    {n_post_only:>6,}  ({n_post_only/n_dropped*100:.1f}% of dropouts)')
print(f'    Pre only  (missing post):   {n_pre_only:>6,}  ({n_pre_only/n_dropped*100:.1f}% of dropouts)')
print()
print('DIFFERENTIAL ATTRITION')
print(f'  Exposed   panel rate: {in_panel_rate_exp:.1f}%')
print(f'  Unexposed panel rate: {in_panel_rate_unexp:.1f}%')
diff = in_panel_rate_exp - in_panel_rate_unexp
flag = 'CLEAN' if abs(diff) < 3 else 'FLAG'
print(f'  Difference: {diff:+.1f} pp  [{flag}]')
print()
print('IMPLICATIONS FOR PAPER')
print('  Dominant dropout cause: silent users (lurkers / deleted accounts).')
print('  Second: missing pre-period baseline — consistent with November-heavy')
print('  exposure timing (most users had little pre-window activity before anchor).')
print('  Pre-only dropout (no post) is the smallest group — reassuring.')
if abs(diff) < 3:
    print('  Attrition is non-differential: selection into panel does not confound.')
else:
    print('  Differential attrition detected — investigate and address in paper.')
print('=' * 65)